# Colab training launcher
Runtime → Change runtime type → **T4 GPU** before running.
Notebooks contain no logic: clone → install → run scripts → download results.

In [1]:
REPO = "https://github.com/xi2618zh-s/spatial-graph-recommendation.git"  # ← 你的真实地址

from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content")
!rm -rf spatial-graph-recommendation experiments
!git clone {REPO}
%cd /content/spatial-graph-recommendation
!pip -q install -r requirements.txt

PERSIST = "/content/drive/MyDrive/sgr_experiments"
os.makedirs(PERSIST, exist_ok=True)
!rm -rf experiments && ln -s {PERSIST} experiments

os.makedirs("data/processed", exist_ok=True)
!cp /content/drive/MyDrive/sgr_data/poi_coords.csv data/processed/
!cp /content/drive/MyDrive/sgr_data/train_sequences.pkl data/processed/

print("=== 自检 ===")
!ls -l | grep experiments
!ls -lh data/processed/
!ls scripts/
!nvidia-smi | head -5

Mounted at /content/drive
Cloning into 'spatial-graph-recommendation'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 97 (delta 25), reused 92 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 2.88 MiB | 17.27 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/spatial-graph-recommendation
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 98.0 MB/s eta 0:00:00
=== 自检 ===
lrwxrwxrwx 1 root root   38 Aug  2 06:48 experiments -> /content/drive/MyDrive/sgr_experiments
total 3.9M
-rw------- 1 root root 1.4M Aug  2 06:48 poi_coords.csv
-rw------- 1 root root 2.6M Aug  2 06:48 train_sequences.pkl
prepare_data.py  run_baselines.py  train.py
Sun Aug  2 06:48:51 2026       
+---------------------------------

In [ ]:
# CPU baselines (~10-20 min): Popularity + ItemCF
!python scripts/run_baselines.py

In [ ]:
# MF-BPR (GPU)
!python scripts/train.py --config configs/mf_gowalla.yaml

In [2]:
# LightGCN (GPU) — the reproduction target: Recall@20 ~= 0.183
!python scripts/train.py --config configs/lightgcn_gowalla.yaml --resume

device: cuda
GowallaData: 29858 users, 40981 items, 810128 train pairs, 217242 test pairs
[build normalized adjacency] ...
[build normalized adjacency] done in 0.1s
/content/spatial-graph-recommendation/src/models/lightgcn.py:19: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, val, m.shape).coalesce()
resumed from epoch 460 (best so far: recall@20=0.1772 @ 440)
epoch  461 | bpr_loss 0.0060
epoch  462 | bpr_loss 0.0059
epoch  463 | bpr_loss 0.0058
epoch  464 | bpr_loss 0.0060
epoch  465 | bpr_loss 0.0059
epoch  466 | bpr_loss 0.0060
epoch  467 | bpr_loss 0.0058
epoch  468 | bpr_loss 0.0059
epoch  469 | bpr_

In [ ]:
# Package logs + results for download (Colab storage is ephemeral!)
!zip -r run_artifacts.zip experiments/
from google.colab import files; files.download('run_artifacts.zip')

In [ ]:
QUEUE = [
    "configs/spatial_lightgcn_gowalla.yaml",   # 主实验:空间增强
    "configs/sasrec_gowalla.yaml",             # 序列召回
    "configs/ablation_spatial_lam0.1.yaml",    # 消融:λ敏感性
    "configs/ablation_spatial_lam0.5.yaml",
    "configs/ablation_spatial_k20.yaml",       # 消融:k敏感性
]
for cfg in QUEUE:
    print(f"\n{'='*60}\nRUN: {cfg}\n{'='*60}")
    !python scripts/train.py --config {cfg} --resume


RUN: configs/spatial_lightgcn_gowalla.yaml
device: cuda
GowallaData: 29858 users, 40981 items, 810128 train pairs, 217242 test pairs
[build combined interaction+spatial adjacency] ...
spatial graph: 447797 edges (k=10, max_dist=100km, sigma=0.21km [auto-median])
[build combined interaction+spatial adjacency] done in 8.3s
/content/spatial-graph-recommendation/src/models/lightgcn.py:19: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, val, m.shape).coalesce()
no checkpoint found, starting fresh
epoch    1 | bpr_loss 0.4225
epoch    2 | bpr_loss 0.1788
epoch    3 | bpr_loss 0.1379
epoch    4 | bpr_loss 0.1188